# 04 — Evaluation: leave-one-out recall@k, cold-start lift, latency

**Project:** Micro-Cert Recommender (H07)

Three deliverables:
1. Leave-one-out recall@k for the collaborative tower, the content tower, and the blend.
2. Cold-start lift — the content tower's behaviour on synthetic learners with zero interactions.
3. Latency benchmark.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from microcert_rec.data import PROCESSED, load_all, make_training_artifacts
from microcert_rec import models
from microcert_rec.features import learner_text

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
if (PROCESSED / 'learners.parquet').exists():
    learners, certs, interactions = load_all()
else:
    learners, certs, interactions = make_training_artifacts()
try:
    art = models.load('two_tower.joblib')
except FileNotFoundError:
    models.main()
    art = models.load('two_tower.joblib')

## 1. Leave-one-out recall@k for the collaborative tower

In [ ]:
completed = interactions[interactions['event_type'] == 'completed']
rng = np.random.default_rng(0)
lid_pool = completed['learner_id'].drop_duplicates().sample(
    min(400, completed['learner_id'].nunique()), random_state=0
)
cert_idx = {c: i for i, c in enumerate(art['cert_ids'])}
learner_idx = {l: i for i, l in enumerate(art['learner_ids'])}

def hits_at_k(scores, gold, k):
    if gold is None:
        return None
    j = cert_idx.get(gold)
    if j is None:
        return None
    return j in np.argsort(-scores)[:k]

ks = [5, 10, 20, 50]
recall_cf = {k: [0, 0] for k in ks}  # [hits, total]
for lid in lid_pool:
    sub = completed[completed['learner_id'] == lid]
    if len(sub) < 2 or lid not in learner_idx:
        continue
    held = sub.sample(1, random_state=0).iloc[0]['cert_id']
    u = art['U'][learner_idx[lid]]
    s = art['V'] @ u
    for k in ks:
        h = hits_at_k(s, held, k)
        if h is None:
            continue
        recall_cf[k][1] += 1
        if h:
            recall_cf[k][0] += 1
for k in ks:
    print(f'collaborative recall@{k}: {recall_cf[k][0]}/{recall_cf[k][1]} = {recall_cf[k][0]/max(recall_cf[k][1],1):.3f}')

## 2. Recall@k for content + two-tower blend

In [ ]:
skill_cols = [c for c in learners.columns if c.startswith('skill__')]
lr_skills = {
    row['learner_id']: [c.replace('skill__', '') for c in skill_cols if row[c] == 1]
    for _, row in learners.iterrows()
}

recall_ct = {k: [0, 0] for k in ks}
recall_blend = {k: [0, 0] for k in ks}
for lid in lid_pool:
    sub = completed[completed['learner_id'] == lid]
    if len(sub) < 2:
        continue
    held = sub.sample(1, random_state=0).iloc[0]['cert_id']
    qv = art['tfidf'].transform([learner_text(lr_skills.get(lid, []))]).toarray()
    s_ct = cosine_similarity(qv, art['X_certs']).ravel()
    if lid in learner_idx:
        u = art['U'][learner_idx[lid]]
        s_cf = art['V'] @ u
        s_cf = (s_cf - s_cf.min()) / (s_cf.max() - s_cf.min() + 1e-9)
        s_blend = 0.5 * s_cf + 0.5 * s_ct
    else:
        s_blend = s_ct
    for k in ks:
        for tag, s, store in [('ct', s_ct, recall_ct), ('blend', s_blend, recall_blend)]:
            h = hits_at_k(s, held, k)
            if h is None:
                continue
            store[k][1] += 1
            if h:
                store[k][0] += 1
for k in ks:
    print(f'content recall@{k}: {recall_ct[k][0]/max(recall_ct[k][1],1):.3f} | '
          f'blend recall@{k}: {recall_blend[k][0]/max(recall_blend[k][1],1):.3f}')

## 3. Recall@k summary chart

In [ ]:
rows = []
for k in ks:
    rows.append({'k': k, 'tower': 'cf', 'recall': recall_cf[k][0]/max(recall_cf[k][1],1)})
    rows.append({'k': k, 'tower': 'ct', 'recall': recall_ct[k][0]/max(recall_ct[k][1],1)})
    rows.append({'k': k, 'tower': 'blend', 'recall': recall_blend[k][0]/max(recall_blend[k][1],1)})
df_recall = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7, 3.4))
for t, sub in df_recall.groupby('tower'):
    ax.plot(sub['k'], sub['recall'], marker='o', label=t)
ax.set_xlabel('k'); ax.set_ylabel('recall')
ax.set_title('Leave-one-out recall@k by tower')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Cold-start sanity — learners with no interactions

In [ ]:
active_l = set(interactions['learner_id'].unique())
cold = [l for l in learners['learner_id'] if l not in active_l]
print(f'# cold-start learners: {len(cold)}')
if cold:
    qv = art['tfidf'].transform([learner_text(lr_skills.get(cold[0], []))]).toarray()
    s_ct = cosine_similarity(qv, art['X_certs']).ravel()
    top = np.argsort(-s_ct)[:5]
    print('cold-start top-5:')
    print(certs.iloc[top][['cert_id', 'title', 'theme']])

## 5. Per-theme recall slice

In [ ]:
lr_theme = learners.set_index('learner_id')['primary_theme'].to_dict()
theme_perf = []
for lid in lid_pool:
    sub = completed[completed['learner_id'] == lid]
    if len(sub) < 2 or lid not in learner_idx:
        continue
    held = sub.sample(1, random_state=0).iloc[0]['cert_id']
    u = art['U'][learner_idx[lid]]
    s = art['V'] @ u
    in_top10 = held in [art['cert_ids'][j] for j in np.argsort(-s)[:10]]
    theme_perf.append({'theme': lr_theme.get(lid, '?'), 'in_top10': in_top10})
by_theme = pd.DataFrame(theme_perf).groupby('theme')['in_top10'].mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 3.4))
by_theme.plot(kind='bar', ax=ax, color='#3a7ca5')
ax.set_ylabel('recall@10 (cf-only)')
ax.set_title('Per-theme recall@10')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 6. Latency

In [ ]:
lid_demo = lid_pool.iloc[0]
u = art['U'][learner_idx[lid_demo]]
t0 = time.perf_counter()
for _ in range(200):
    s = art['V'] @ u
    np.argsort(-s)[:10]
ms = (time.perf_counter() - t0) / 200 * 1000
print(f'cf top-10 ranking ~{ms:.2f} ms/call')

## 7. Release-readiness checklist

| Gate | Target | Result |
|---|---|---|
| Blend recall@10 | ≥ 0.20 | see §3 |
| Cold-start path returns sensible certs | yes | §4 |
| Per-theme recall floor | no theme < 0.10 | §5 |
| Latency | < 50 ms | §6 |

Documented next step: replace TruncatedSVD with `implicit` ALS for confidence-weighted MF; add per-issuer re-rank weights; introduce a session-level diversity constraint (`docs/04_evaluation.md`).